<h1 align="center">🧩 Prolog: First-Order Logic, Unification & Resolution</h1>

<p align="center"><i>From propositional logic to objects and relations: formalize a domain in FOL, unify terms, and prove theorems by refutation, all in SWI-Prolog.</i></p>

<p align="center">
  <img src="https://img.shields.io/badge/Course-Knowledge%20Representation%20%26%20Reasoning-6f42c1" />
  <img src="https://img.shields.io/badge/Prolog-SWI--Prolog-red" />
  <img src="https://img.shields.io/badge/Level-Intermediate-yellow" />
</p>

<p align="center">
  <a href="https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/2-prolog/prolog_logic_programming.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</p>

---

## 📌 At a Glance

| | |
|---|---|
| **Topics** | FOL · Quantifiers · Prolog Facts & Rules · Unification · MGU · Occurs-Check · Resolution Refutation |
| **Covers** | First-Order Logic · Unification & Substitution · Resolution Refutation |
| **Tools** | SWI-Prolog, called from Python via `subprocess` |
| **Run time** | 1 to 2 minutes (mostly the apt install) |

## 🗂️ Contents
1. [First-Order Logic in Prolog](#part-1)
2. [Unification & Substitution](#part-2)
3. [Resolution Refutation](#part-3)
4. [Summary](#summary)

---

**Setup** (only needed once, e.g. on a fresh Colab runtime). This installs SWI-Prolog and defines a small helper `prolog(program, goal)` that runs Prolog code and prints the result:

In [1]:
!apt-get install -y -qq swi-prolog-nox > /dev/null

In [2]:
import subprocess

PRELUDE = r"""
show(G) :- \+ \+ ( numbervars(G,0,_), format('?- ~W.~n',[G,[numbervars(true),quoted(true)]]) ).
check(G) :- show(G), ( call(G) -> writeln('   true') ; writeln('   false') ).
all(T,G) :- show(G), findall(T,G,L), format('   ~w~n',[L]).
"""

def prolog(program, goal):
    """Write PRELUDE + program to kb.pl, run `goal` in SWI-Prolog, print the output."""
    with open('kb.pl', 'w') as f:
        f.write(PRELUDE + program)
    r = subprocess.run(['swipl', '-q', '-s', 'kb.pl', '-g', goal, '-t', 'halt'],
                       capture_output=True, text=True, timeout=30)
    print((r.stdout + r.stderr).rstrip())

<a id="part-1"></a>

## 📚 Part 1 · First-Order Logic in Prolog

> 🎯 **Goal:** Translate English into FOL, then encode the same knowledge as Prolog facts and rules.

FOL extends propositional logic with **predicates**, **functions** and **quantifiers** (∀, ∃), so we can talk about objects and their relations.

### 1.1 English → FOL

| # | English | First-Order Logic | Prolog |
|---|---|---|---|
| 1 | All humans are mortal | $\forall x\,(Human(x) \rightarrow Mortal(x))$ | `mortal(X) :- human(X).` |
| 2 | Socrates is a human | $Human(socrates)$ | `human(socrates).` |
| 3 | Some student loves AI | $\exists x\,(Student(x) \land Loves(x, ai))$ | `student(X), loves(X, ai).` (a query) |

A Prolog **rule** `head :- body` reads "body implies head", and its variables are implicitly universally quantified. A **query** with variables is implicitly existential: Prolog searches for a witness.

### 1.2 Encode and query the domain

In [3]:
program = r"""
human(socrates).
human(plato).
student(ali).
loves(ali, ai).
mortal(X) :- human(X).      % forall x: human(x) -> mortal(x)
"""

goal = """
check(mortal(socrates)),
check(loves(ali, ai)),
all(X, mortal(X)),
all(X, (student(X), loves(X, ai)))
"""
prolog(program, goal)

?- mortal(socrates).
   true
?- loves(ali,ai).
   true
?- mortal(A).
   [socrates,plato]
?- student(A),loves(A,ai).
   [ali]

### 1.3 Universal vs existential

- **Universal** ($\forall$): a *rule*. It must hold for every object, so it is written with a variable in the head.
- **Existential** ($\exists$): a *query* (or a fact naming a witness). We only need one example.

### 1.4 Practice tasks

**Task 1.** "Every dog has an owner": $\forall x\,(Dog(x) \rightarrow \exists y\,Owns(y, x))$

In Prolog the $\exists y$ becomes an anonymous variable `_` inside the rule body.

In [4]:
program = r"""
dog(rex).
dog(bella).
owns(ali, rex).
owns(sara, bella).

has_owner(D) :- dog(D), owns(_, D).     % exists y: owns(y, D)
owner_of(D, O) :- dog(D), owns(O, D).
"""

goal = """
check(has_owner(rex)),
all(D, has_owner(D)),
all(D-O, owner_of(D, O))
"""
prolog(program, goal)

?- has_owner(rex).
   true
?- has_owner(A).
   [rex,bella]
?- owner_of(A,B).
   [rex-ali,bella-sara]

**Task 2.** Three sentences about a university: *students, courses, enrolled*.

In [5]:
program = r"""
student(ali).   student(sara).   student(bilal).
course(krr).    course(ai).
enrolled(ali, krr).
enrolled(sara, krr).
enrolled(sara, ai).

% A classmate shares at least one course with another student.
classmates(A, B) :- enrolled(A, C), enrolled(B, C), A \= B.
% A student who is not enrolled in anything.
idle(S) :- student(S), \+ enrolled(S, _).
"""

goal = """
all(S, enrolled(S, krr)),
all(C, enrolled(sara, C)),
all(A-B, classmates(A, B)),
all(S, idle(S))
"""
prolog(program, goal)

?- enrolled(A,krr).
   [ali,sara]
?- enrolled(sara,A).
   [krr,ai]
?- classmates(A,B).
   [ali-sara,sara-ali]
?- idle(A).
   [bilal]

**Task 3.** "No fish is a mammal": $\forall x\,(Fish(x) \rightarrow \neg Mammal(x))$

Prolog has no true logical negation. Instead it uses **negation as failure** (`\+`): `\+ G` succeeds when `G` cannot be proved. The sentence is a *constraint*, so we encode it as a check that looks for a violation.

In [6]:
program = r"""
fish(salmon).
fish(shark).
mammal(dolphin).
mammal(whale).

% Constraint: no fish is a mammal  ==  there is no x with fish(x) and mammal(x)
violation(X) :- fish(X), mammal(X).
consistent :- \+ violation(_).

not_mammal(X) :- fish(X), \+ mammal(X).
"""

goal = """
check(mammal(salmon)),
all(X, not_mammal(X)),
check(consistent)
"""
prolog(program, goal)

?- mammal(salmon).
   false
?- not_mammal(A).
   [salmon,shark]
?- consistent.
   true

> 💡 **Key Takeaway:** A Prolog *rule* encodes a universally quantified implication, a *query with variables* encodes an existential, and `\+` is negation as failure (not classical $\neg$).

---
<a id="part-2"></a>

## 🔗 Part 2 · Unification & Substitution

> 🎯 **Goal:** Find a substitution that makes two terms identical, and compute the **most general unifier (MGU)**.

A **substitution** replaces variables by terms, e.g. $\{X/ali,\ Y/ai\}$. A **unifier** of two expressions makes them identical; the **MGU** commits to as little as possible. Unification is the engine behind resolution **and** Prolog itself: every `=` and every rule-head match is a unification.

### 2.1 Built-in unification with `=`

In [7]:
goal = """
% loves(X, ai)  vs  loves(ali, Y)
loves(X, ai) = loves(ali, Y),
format('MGU: X = ~w, Y = ~w~n', [X, Y])
"""
prolog('', goal)

MGU: X = ali, Y = ai

### 2.2 Practice tasks with the built-in

**Task 1.** Unify `knows(X, X)` with `knows(ali, Y)`. The first argument binds $X = ali$; the repeated $X$ then forces $Y = ali$ as well.

**Task 2.** A failing case: two different constants never unify.

**Task 3.** `f(X, g(Y))` with `f(a, g(b))`, nested terms.

In [8]:
goal = """
( knows(X, X) = knows(ali, Y)
  -> format('1) knows(X,X) ~~ knows(ali,Y):  X = ~w, Y = ~w~n', [X, Y]) ; writeln('1) fails') ),

( loves(ali, ai) = loves(bob, ai)
  -> writeln('2) unified') ; writeln('2) loves(ali,ai) ~ loves(bob,ai):  FAIL (ali \\\\= bob)') ),

( f(A, g(B)) = f(a, g(b))
  -> format('3) f(A,g(B)) ~~ f(a,g(b)):  A = ~w, B = ~w~n', [A, B]) ; writeln('3) fails') )
"""
prolog('', goal)

1) knows(X,X) ~ knows(ali,Y):  X = ali, Y = ali
2) loves(ali,ai) ~ loves(bob,ai):  FAIL (ali \= bob)
3) f(A,g(B)) ~ f(a,g(b)):  A = a, B = b

### 2.3 Occurs-check

Unifying `X` with `f(X)` would create an infinite term. Standard Prolog's `=` skips this check for speed; `unify_with_occurs_check/2` performs it.

In [9]:
goal = """
( unify_with_occurs_check(X, f(X))
  -> writeln('X unifies with f(X)')
  ;  writeln('X ~ f(X): FAIL with occurs-check (X occurs inside f(X))') ),

( unify_with_occurs_check(X1, f(Y1))
  -> format('X ~~ f(Y) is fine: X = f(Y)~n', []) ; true )
"""
prolog('', goal)

X ~ f(X): FAIL with occurs-check (X occurs inside f(X))
X ~ f(Y) is fine: X = f(Y)

### 2.4 Our own unification algorithm

Prolog does this for us, but to see *how* it works we re-implement the unification algorithm explicitly. Variables are wrapped as `v(Name)` so they are treated as ordinary data, and the substitution is a list of `Var-Term` pairs.

In [10]:
program = r"""
is_var(v(_)).

% unify(X, Y, SubstIn, SubstOut)
unify(X, Y, S, S) :- X == Y, !.
unify(X, Y, S0, S) :- is_var(X), !, unify_var(X, Y, S0, S).
unify(X, Y, S0, S) :- is_var(Y), !, unify_var(Y, X, S0, S).
unify(X, Y, S0, S) :-
    compound(X), compound(Y),
    X =.. [F|As], Y =.. [F|Bs],
    length(As, N), length(Bs, N),
    unify_list(As, Bs, S0, S).

unify_list([], [], S, S).
unify_list([A|As], [B|Bs], S0, S) :- unify(A, B, S0, S1), unify_list(As, Bs, S1, S).

unify_var(V, X, S0, S) :- memberchk(V-T, S0), !, unify(T, X, S0, S).      % already bound
unify_var(V, X, _, _)  :- occurs(V, X), !, fail.                          % occurs-check
unify_var(V, X, S0, [V-X|S0]).

occurs(V, T) :- T == V, !.
occurs(V, T) :- compound(T), T =.. [_|As], member(A, As), occurs(V, A).

mgu(T1, T2) :-
    (   unify(T1, T2, [], S0) -> reverse(S0, S), format('MGU: ~w~n', [S])
    ;   writeln('MGU: none (unification fails)') ).
"""

goal = """
mgu(loves(v('X'), ai),       loves(ali, v('Y'))),
mgu(knows(v('X'), v('X')),   knows(ali, v('Y'))),
mgu(f(v('X'), g(v('Y'))),    f(a, g(b))),
mgu(loves(ali, ai),          loves(bob, ai)),
mgu(v('X'),                  f(v('X')))
"""
prolog(program, goal)

MGU: [v(X)-ali,v(Y)-ai]
MGU: [v(X)-ali,v(Y)-ali]
MGU: [v(X)-a,v(Y)-b]
MGU: none (unification fails)
MGU: none (unification fails)

Reading the output: `knows(X, X)` vs `knows(ali, Y)` first binds $X/ali$; then $X$ and $Y$ are unified again, so $X$ is dereferenced to `ali` and we get $Y/ali$. The last case fails because of the occurs-check.

> 💡 **Key Takeaway:** The **MGU** is the least-committal substitution that makes two terms equal. Prolog runs unification on every goal, so it is the real engine behind logic programming.

---
<a id="part-3"></a>

## 🔬 Part 3 · Resolution Refutation

> 🎯 **Goal:** Prove $KB \models \alpha$ by **refutation**: add $\neg\alpha$ to the KB and derive the **empty clause** $\square$ (a contradiction).

1. Write the KB in clause form (a clause is a set of literals).
2. Add the negated query as a new clause.
3. Resolve pairs of clauses containing complementary literals ($L$ and $\neg L$).
4. If $\square$ appears, the query is **entailed**. If no new clauses can be produced, it is **not**.

Literals are Prolog atoms; negation is written `~p` using a prefix operator.

### 3.1 Resolution prover

In [11]:
program = r"""
:- op(200, fy, ~).

negate(~A, A) :- !.
negate(A, ~A).

% resolve(Ci, Cj, R): R is a resolvent of Ci and Cj (clauses are lists of literals)
resolve(Ci, Cj, R) :-
    member(L, Ci), negate(L, NL), member(NL, Cj),
    delete(Ci, L, Ci1), delete(Cj, NL, Cj1),
    append(Ci1, Cj1, R0), sort(R0, R).

% saturate(Clauses, Round): resolve all pairs until [] appears or nothing new is derived
saturate(Cs, _) :-
    member(A, Cs), member(B, Cs), A \== B, resolve(A, B, []), !,
    format('  ~w + ~w  =>  [] (empty clause)~n', [A, B]).
saturate(Cs, N) :-
    findall(R, (member(A, Cs), member(B, Cs), A \== B, resolve(A, B, R)), Rs0),
    sort(Rs0, Rs), subtract(Rs, Cs, New),
    New \== [],
    forall(member(C, New), format('  round ~w: derived ~w~n', [N, C])),
    append(Cs, New, Cs2), N1 is N + 1,
    saturate(Cs2, N1).

% entails(KB, Query): proof by refutation
entails(KB, Query) :-
    negate(Query, NQ),
    maplist(sort, KB, KB1),
    format('Negated goal added: ~w~n', [[NQ]]),
    (   saturate([[NQ]|KB1], 1)
    ->  format('KB entails ~w: True~n', [Query])
    ;   format('KB entails ~w: False (no new clauses, no contradiction)~n', [Query]) ).
"""

goal = """
% KB: (P -> Q) and P, written as clauses.  Prove Q.
entails([[~p, q], [p]], q)
"""
prolog(program, goal)

Negated goal added: [~q]
  round 1: derived [q]
  round 1: derived [~p]
  [~q] + [q]  =>  [] (empty clause)
KB entails q: True

Here $P \rightarrow Q$ becomes the clause $\{\neg P, Q\}$. Adding $\{\neg Q\}$ and resolving gives $\{\neg P\}$, which clashes with $\{P\}$ to yield $\square$. The contradiction means $Q$ follows from the KB.

### 3.2 Practice tasks

In [12]:
# `program` still holds the resolution prover defined in the previous cell
tasks = r"""
% Task 1: {P or Q, ~P} entails Q
writeln('--- Task 1: {p v q, ~p} |= q'),
entails([[p, q], [~p]], q),

% Task 2: add (Q -> R) and prove R from P
writeln(''),
writeln('--- Task 2: {p, p->q, q->r} |= r'),
entails([[p], [~p, q], [~q, r]], r),

% Task 3: a case with NO proof
writeln(''),
writeln('--- Task 3: {p v q} |= p ?'),
entails([[p, q]], p)
"""
prolog(program, tasks)

--- Task 1: {p v q, ~p} |= q
Negated goal added: [~q]
  round 1: derived [p]
  round 1: derived [q]
  [~q] + [q]  =>  [] (empty clause)
KB entails q: True

--- Task 2: {p, p->q, q->r} |= r
Negated goal added: [~r]
  round 1: derived [q]
  round 1: derived [r,~p]
  round 1: derived [~q]
  [q] + [~q]  =>  [] (empty clause)
KB entails r: True

--- Task 3: {p v q} |= p ?
Negated goal added: [~p]
  round 1: derived [q]
KB entails p: False (no new clauses, no contradiction)

Task 3 returns **False** because $\{P \lor Q\}$ does not force $P$: the assignment $P = false,\ Q = true$ satisfies the KB but falsifies the query, so no refutation exists.

### 3.3 Cross-check with propositional logic (SymPy)

The same entailments can be verified with the truth-table / SymPy tools from the Knowledge Bases & Propositional Logic notebook:

In [13]:
from sympy import symbols
from sympy.logic.inference import entails

P, Q, R = symbols('P Q R')
print('{P | Q, ~P}    |= Q :', entails(Q, [P | Q, ~P]))
print('{P, P>>Q, Q>>R} |= R :', entails(R, [P, P >> Q, Q >> R]))
print('{P | Q}         |= P :', entails(P, [P | Q]))

{P | Q, ~P}    |= Q : True
{P, P>>Q, Q>>R} |= R : True
{P | Q}         |= P : False

### 3.4 Prolog *is* resolution

Prolog's own engine is a restricted form of resolution (**SLD resolution**) on Horn clauses. When we ask `?- mortal(socrates).`, Prolog negates the goal, resolves it with the rule `mortal(X) :- human(X)` using the MGU $\{X/socrates\}$, then with the fact `human(socrates)`, and reaches the empty clause. Parts 1, 2 and 3 of today's notebook are therefore three views of one machine.

> 💡 **Key Takeaway:** **Refutation** turns "is it entailed?" into "can we derive a contradiction?". Reaching $\square$ proves entailment; saturating without $\square$ proves non-entailment.

---
<a id="summary"></a>
## ✅ Summary

| Part | Topic | What you learned |
|---|---|---|
| 1 | First-Order Logic | Predicates, quantifiers, English→FOL, facts & rules in Prolog, negation as failure |
| 2 | Unification | Substitution, MGU, occurs-check, own unifier in Prolog |
| 3 | Resolution | Clause form, refutation, empty clause, entailment / non-entailment |

## 🏋️ Practice Exercises
1. Translate "Every student has a teacher" to FOL and Prolog.
2. Find the MGU of `p(X, f(Y), Z)` and `p(a, f(b), W)` with the Prolog unifier above.
3. Use `entails/2` to prove $R$ from $\{P \lor Q,\ \neg P \lor R,\ \neg Q \lor R\}$ (proof by cases).

<p align="center"><i>⬅️ Back to First-Order Logic · Next: Protégé ➡️</i></p>